# falsify-eval · 60-second quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spalsh-spec/falsify-eval/blob/main/notebooks/quickstart.ipynb)

**No install required.** This Colab is the lowest-friction path to actually running the four-null gate. Run cell 1, run cell 2, see the verdict.

If you want sliders without writing code, try the [Mira Playground](https://spalsh-spec.github.io/falsify-eval/play.html) instead.

In [ ]:
!pip install -q git+https://github.com/spalsh-spec/falsify-eval

In [ ]:
from falsify_eval.demo import run
run()

## Try it on YOUR retriever

Replace `my_retriever` with whatever returns top-K document IDs for a query.

In [ ]:
from falsify_eval import four_null_gate
import numpy as np

rng = np.random.default_rng(2026)
labels = [f'L{i}' for i in range(8)]
gold = [labels[i % 8] for i in range(80)]

def my_retriever(q_idx):
    g = gold[q_idx]
    others = [l for l in labels if l != g]
    return [g] + list(rng.choice(others, size=4, replace=False))

retrieved = [my_retriever(i) for i in range(len(gold))]

def recall_at_5(r, g, _rel):
    return 1.0 if g in r[:5] else 0.0

result = four_null_gate(
    retrieved, gold, [3]*len(gold), recall_at_5,
    item_pool=labels, k=5, n_trials=30, tau=0.05, seed=2026,
)
print('GATE:', 'PASS' if result['gate_passes'] else 'FAIL')
print('real:', round(result['real_mean'], 3))
for x in 'ABCD':
    print(f'  delta_{x} = {result["deltas"][x]:+.3f}   {"PASS" if result["passes"][x] else "FAIL"}')

## Where to next

- **Real-world case study:** [CS01 — NFCorpus](https://github.com/spalsh-spec/falsify-eval/blob/main/case_studies/cs01_nfcorpus/CS01_REPORT.md) — 5 minutes to reproduce.
- **Methodology:** [PREPRINT.md](https://github.com/spalsh-spec/falsify-eval/blob/main/PREPRINT.md).
- **Source:** https://github.com/spalsh-spec/falsify-eval (Apache 2.0).